# Dữ liệu và tokenizer

TV1 kiểm tra các tập rồi bàn giao quy ước mã hóa cho TV2 và TV3.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/corpus.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.corpus import records

sample = next(records("train"))
print(sample)

In [ ]:
for split in ("train", "validation", "test"):
    print(split, sum(1 for _ in records(split)))

## Ba cách tách token

Cùng dùng SentencePiece làm thư viện cho hai thuật toán BPE và Unigram để giữ nguyên các bước còn lại. Chạy hàm cho từng ngôn ngữ. Với mức từ, dùng top 8.000 mục kể cả ký hiệu đặc biệt, xây từ điển bằng tần suất trên train và ánh xạ từ hiếm sang UNK.

In [ ]:
def fit_subwords(language, algorithm):
    import sentencepiece as spm

    output = root / "artifacts" / "tokenizers"
    output.mkdir(parents=True, exist_ok=True)
    spm.SentencePieceTrainer.train(
        sentence_iterator=(r[language] for r in records("train")),
        model_prefix=str(output / f"{language}-{algorithm}"), model_type=algorithm,
        vocab_size=8000, pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        normalization_rule_name="identity", character_coverage=1.0,
    )

def word_tokens(text, language):
    if language == "en":
        return text.split()
    from underthesea import word_tokenize
    return word_tokenize(text, format="text").split()

# Khi chuẩn bị huấn luyện:
# for lang in ("en", "vi"):
#     for algorithm in ("bpe", "unigram"):
#         fit_subwords(lang, algorithm)